# Leather Defect Area Calculator
**Model:** Attention U-Net (best_attention_unet.keras)  
**Task:** Predict defect masks → compute total damaged area  
**Classes:** background | color | cut | fold | glue | poke

In [10]:
import os
import glob
import warnings
warnings.filterwarnings('ignore')

import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pandas as pd

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'

import tensorflow as tf
from tensorflow.keras import backend as K

print(f'TensorFlow {tf.__version__}')
print(f'GPUs: {tf.config.list_physical_devices("GPU")}')

TensorFlow 2.20.0
GPUs: []


## 1. Configuration

In [11]:
BASE_DIR   = r'd:\7th sem\Image processing and CV\Leather_defect_detection\leather'
MODEL_PATH = r'd:\7th sem\Image processing and CV\Leather_defect_detection\unet_results\best_attention_unet.keras'

IMG_SIZE     = (256, 256)
NUM_CLASSES  = 6
CLASS_NAMES  = ['background', 'color', 'cut', 'fold', 'glue', 'poke']
DEFECT_TYPES = ['color', 'cut', 'fold', 'glue', 'poke']

CLASS_COLORS = np.array([
    [0,   0,   0  ],   # background – black
    [255, 0,   0  ],   # color      – red
    [0,   255, 0  ],   # cut        – green
    [0,   0,   255],   # fold       – blue
    [255, 255, 0  ],   # glue       – yellow
    [255, 0,   255],   # poke       – magenta
], dtype=np.uint8)

# ---------------------------------------------------------------------------
# Optional: real-world scale
# If you know the physical size of the leather sample, set these to convert
# pixel counts to mm².  Leave as None to report in pixels & percentages only.
# ---------------------------------------------------------------------------
SAMPLE_WIDTH_MM  = None   # e.g. 200  (width of the leather sample in mm)
SAMPLE_HEIGHT_MM = None   # e.g. 200

## 2. Custom objects required to load the model

In [ ]:
FOCAL_GAMMA = 2.0
FOCAL_ALPHA = 0.25
DICE_SMOOTH = 1.0

def focal_loss(y_true, y_pred, gamma=FOCAL_GAMMA, alpha=FOCAL_ALPHA):
    y_true  = tf.cast(tf.reshape(y_true, [-1]), tf.int32)
    y_pred  = tf.reshape(y_pred, [-1, NUM_CLASSES])
    y_pred  = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)
    y_oh    = tf.one_hot(y_true, NUM_CLASSES)
    ce      = -y_oh * tf.math.log(y_pred)
    pt      = tf.reduce_sum(y_oh * y_pred, axis=-1)
    fw      = alpha * tf.pow(1.0 - pt, gamma)
    return tf.reduce_mean(tf.reduce_sum(ce, axis=-1) * fw)

def dice_loss_fn(y_true, y_pred, smooth=DICE_SMOOTH):
    y_true  = tf.cast(tf.reshape(y_true, [-1]), tf.int32)
    y_pred  = tf.reshape(y_pred, [-1, NUM_CLASSES])
    y_oh    = tf.one_hot(y_true, NUM_CLASSES)
    inter   = tf.reduce_sum(y_oh * y_pred, axis=0)
    union   = tf.reduce_sum(y_oh, axis=0) + tf.reduce_sum(y_pred, axis=0)
    dice_pc = (2.0 * inter + smooth) / (union + smooth)
    return 1.0 - tf.reduce_mean(dice_pc[1:])

def combined_loss(y_true, y_pred):
    return 0.5 * focal_loss(y_true, y_pred) + 0.5 * dice_loss_fn(y_true, y_pred)

def mean_iou_metric(y_true, y_pred):
    y_true = tf.cast(tf.reshape(y_true, [-1]), tf.int32)
    y_pred = tf.cast(tf.argmax(tf.reshape(y_pred, [-1, NUM_CLASSES]), axis=-1), tf.int32)
    cm     = tf.cast(tf.math.confusion_matrix(y_true, y_pred, num_classes=NUM_CLASSES), tf.float32)
    diag   = tf.linalg.diag_part(cm)
    denom  = tf.reduce_sum(cm, 1) + tf.reduce_sum(cm, 0) - diag
    iou    = tf.where(denom > 0, diag / denom, tf.zeros_like(diag))
    valid  = tf.cast(denom > 0, tf.float32)
    return tf.math.divide_no_nan(tf.reduce_sum(iou), tf.reduce_sum(valid))

def mean_dice_metric(y_true, y_pred):
    y_true = tf.cast(tf.reshape(y_true, [-1]), tf.int32)
    y_pred = tf.cast(tf.argmax(tf.reshape(y_pred, [-1, NUM_CLASSES]), axis=-1), tf.int32)
    cm     = tf.cast(tf.math.confusion_matrix(y_true, y_pred, num_classes=NUM_CLASSES), tf.float32)
    diag   = tf.linalg.diag_part(cm)
    denom  = tf.reduce_sum(cm, 1) + tf.reduce_sum(cm, 0)
    dice   = tf.where(denom > 0, 2.0 * diag / denom, tf.zeros_like(diag))
    valid  = tf.cast(denom > 0, tf.float32)
    return tf.math.divide_no_nan(tf.reduce_sum(dice), tf.reduce_sum(valid))

CUSTOM_OBJECTS = {
    'combined_loss':    combined_loss,
    'mean_iou_metric':  mean_iou_metric,
    'mean_dice_metric': mean_dice_metric,
}

## 3. Load the trained Attention U-Net

In [ ]:
assert os.path.exists(MODEL_PATH), f'Model not found at: {MODEL_PATH}'

# Keras 3.x resolves built-in layers by module path, ignoring custom_objects.
# Patch Dense.__init__ directly to absorb the unknown quantization_config kwarg
# that was serialized by a newer Keras version.
import keras as _keras
_orig_dense_init = _keras.layers.Dense.__init__

def _patched_dense_init(self, *args, quantization_config=None, **kwargs):
    _orig_dense_init(self, *args, **kwargs)

_keras.layers.Dense.__init__ = _patched_dense_init
try:
    model = tf.keras.models.load_model(MODEL_PATH, custom_objects=CUSTOM_OBJECTS)
finally:
    _keras.layers.Dense.__init__ = _orig_dense_init  # always restore

print(f'Model loaded: {model.name}')
print(f'Input shape : {model.input_shape}')
print(f'Output shape: {model.output_shape}')
print(f'Parameters  : {model.count_params():,}')

## 4. Helper functions

In [ ]:
def preprocess_image(img_path):
    """Load → grayscale → resize → normalize → add batch & channel dims."""
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise FileNotFoundError(img_path)
    img = cv2.resize(img, IMG_SIZE, interpolation=cv2.INTER_LINEAR)
    img = img.astype(np.float32) / 255.0
    return img[..., np.newaxis]          # (256, 256, 1)


def predict_mask(img_tensor):
    """Run inference; return predicted class mask (H, W) int."""
    batch  = img_tensor[np.newaxis, ...]          # (1, 256, 256, 1)
    probs  = model.predict(batch, verbose=0)[0]   # (256, 256, 6)
    return np.argmax(probs, axis=-1).astype(np.uint8)  # (256, 256)


def colorize_mask(mask):
    """Convert integer class mask to RGB colour image."""
    h, w = mask.shape
    rgb  = np.zeros((h, w, 3), dtype=np.uint8)
    for cls_id in range(NUM_CLASSES):
        rgb[mask == cls_id] = CLASS_COLORS[cls_id]
    return rgb


def compute_area(pred_mask, orig_h=None, orig_w=None):
    """
    Compute defect area statistics from a predicted mask.

    Returns a dict:
      total_pixels       – total image pixels
      defect_pixels      – non-background pixels
      defect_pct         – defect_pixels / total_pixels * 100
      per_class_pixels   – dict {class_name: pixel_count}
      per_class_pct      – dict {class_name: percentage}
      defect_mm2         – total defect area in mm² (if scale provided)
    """
    total_px = pred_mask.size
    per_class = {}
    for cls_id, cls_name in enumerate(CLASS_NAMES):
        per_class[cls_name] = int(np.sum(pred_mask == cls_id))

    defect_px  = total_px - per_class['background']
    defect_pct = defect_px / total_px * 100.0

    per_class_pct = {k: v / total_px * 100.0 for k, v in per_class.items()}

    # Real-world area (optional)
    defect_mm2 = None
    if SAMPLE_WIDTH_MM and SAMPLE_HEIGHT_MM and orig_h and orig_w:
        px_per_mm2 = (orig_h * orig_w) / (SAMPLE_HEIGHT_MM * SAMPLE_WIDTH_MM)
        defect_mm2 = defect_px / px_per_mm2

    return {
        'total_pixels':     total_px,
        'defect_pixels':    defect_px,
        'defect_pct':       defect_pct,
        'per_class_pixels': per_class,
        'per_class_pct':    per_class_pct,
        'defect_mm2':       defect_mm2,
    }

## 5. Discover test images

In [ ]:
image_records = []   # list of (img_path, defect_type)

for dtype in DEFECT_TYPES:
    test_dir = os.path.join(BASE_DIR, 'test', dtype)
    paths    = sorted(glob.glob(os.path.join(test_dir, '*.png')))
    for p in paths:
        image_records.append({'path': p, 'defect_type': dtype})
    print(f'  {dtype:8s}: {len(paths):3d} images')

print(f'\nTotal images to process: {len(image_records)}')

## 6. Run predictions and calculate damaged area

In [ ]:
results = []

for i, rec in enumerate(image_records):
    img_path    = rec['path']
    defect_type = rec['defect_type']
    img_name    = os.path.basename(img_path)

    # Read original for size info (used only if real-world scale is configured)
    orig_img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    orig_h, orig_w = (orig_img.shape if orig_img is not None else (None, None))

    img_tensor = preprocess_image(img_path)
    pred_mask  = predict_mask(img_tensor)
    area_stats = compute_area(pred_mask, orig_h, orig_w)

    results.append({
        'image':         img_name,
        'defect_type':   defect_type,
        'total_px':      area_stats['total_pixels'],
        'defect_px':     area_stats['defect_pixels'],
        'defect_pct':    round(area_stats['defect_pct'], 4),
        'defect_mm2':    area_stats['defect_mm2'],
        **{f'px_{k}': v for k, v in area_stats['per_class_pixels'].items()},
        **{f'pct_{k}': round(v, 4) for k, v in area_stats['per_class_pct'].items()},
        '_pred_mask':    pred_mask,
        '_img_tensor':   img_tensor,
    })

    if (i + 1) % 10 == 0 or (i + 1) == len(image_records):
        print(f'  Processed {i+1}/{len(image_records)} — '
              f'{img_name} | defect: {area_stats["defect_pct"]:.2f}%')

print('\nDone.')

## 7. Summary table

In [ ]:
# Build a clean DataFrame (drop internal columns)
report_cols = ['image', 'defect_type', 'total_px', 'defect_px', 'defect_pct'] + \
              [f'px_{c}' for c in CLASS_NAMES[1:]] + \
              [f'pct_{c}' for c in CLASS_NAMES[1:]]
if results[0]['defect_mm2'] is not None:
    report_cols.insert(5, 'defect_mm2')

df = pd.DataFrame([{k: r[k] for k in report_cols} for r in results])

print('=== Per-image area report (first 10 rows) ===')
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 120)
display(df.head(10))

In [ ]:
# Per-defect-type summary
summary = df.groupby('defect_type').agg(
    images      = ('image',      'count'),
    mean_defect_pct = ('defect_pct', 'mean'),
    min_defect_pct  = ('defect_pct', 'min'),
    max_defect_pct  = ('defect_pct', 'max'),
    total_defect_px = ('defect_px',  'sum'),
).reset_index()

# Overall row
overall = pd.DataFrame([{
    'defect_type':       'TOTAL',
    'images':            df.shape[0],
    'mean_defect_pct':   df['defect_pct'].mean(),
    'min_defect_pct':    df['defect_pct'].min(),
    'max_defect_pct':    df['defect_pct'].max(),
    'total_defect_px':   df['defect_px'].sum(),
}])
summary = pd.concat([summary, overall], ignore_index=True)

print('=== Per-defect-type area summary ===')
display(summary.round(4))

In [ ]:
# Grand total damage area
total_images   = len(results)
total_defect_px = df['defect_px'].sum()
total_all_px    = df['total_px'].sum()
overall_pct     = total_defect_px / total_all_px * 100

print('=' * 52)
print('  TOTAL DAMAGED LEATHER AREA')
print('=' * 52)
print(f'  Images analysed   : {total_images}')
print(f'  Total pixels      : {total_all_px:,}')
print(f'  Defect pixels     : {total_defect_px:,}')
print(f'  Overall damage    : {overall_pct:.2f}%')

if results[0]['defect_mm2'] is not None:
    total_mm2 = df['defect_mm2'].sum()
    print(f'  Total defect area : {total_mm2:.1f} mm²  ({total_mm2/100:.2f} cm²)')

print()
print('  Per-class breakdown (across all images):')
for cls in CLASS_NAMES[1:]:
    px  = df[f'px_{cls}'].sum()
    pct = px / total_all_px * 100
    bar = '#' * int(pct * 2)
    print(f'    {cls:10s}: {px:8,} px  ({pct:5.2f}%)  {bar}')
print('=' * 52)

## 8. Visual charts

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Leather Defect Area Analysis — Attention U-Net', fontsize=14, fontweight='bold')

# --- Chart 1: mean defect % per defect type ---
ax = axes[0]
sub = summary[summary['defect_type'] != 'TOTAL']
colors_bar = ['#e74c3c', '#2ecc71', '#3498db', '#f1c40f', '#9b59b6']
bars = ax.bar(sub['defect_type'], sub['mean_defect_pct'], color=colors_bar, edgecolor='white')
ax.set_title('Mean Defect Area (%) per Defect Type', fontsize=12)
ax.set_ylabel('Mean % of image area')
ax.set_ylim(0, max(sub['mean_defect_pct'].max() * 1.3, 1))
ax.grid(axis='y', alpha=0.3)
for bar in bars:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width() / 2, h + 0.1, f'{h:.2f}%',
            ha='center', va='bottom', fontsize=9)

# --- Chart 2: total defect pixels per class (pie) ---
ax = axes[1]
class_totals = [df[f'px_{c}'].sum() for c in CLASS_NAMES[1:]]
norm_colors  = [CLASS_COLORS[i+1] / 255.0 for i in range(len(CLASS_NAMES[1:]))]
wedge_labels = [f'{n}\n({v/sum(class_totals)*100:.1f}%)'
                for n, v in zip(CLASS_NAMES[1:], class_totals)]
wedges, _ = ax.pie(class_totals, labels=wedge_labels,
                   colors=norm_colors, startangle=90,
                   wedgeprops=dict(edgecolor='white', linewidth=1.2))
ax.set_title('Defect Pixel Distribution by Class', fontsize=12)

# --- Chart 3: per-image defect % scatter ---
ax = axes[2]
for i, dtype in enumerate(DEFECT_TYPES):
    sub_df = df[df['defect_type'] == dtype]
    ax.scatter(range(len(sub_df)), sub_df['defect_pct'],
               label=dtype, alpha=0.75, s=40, color=np.array(CLASS_COLORS[i+1]) / 255.0)
ax.set_title('Per-image Defect Area (%)', fontsize=12)
ax.set_xlabel('Image index within defect type')
ax.set_ylabel('Defect area (%)')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 9. Visual prediction samples (2 images per defect type)

In [ ]:
SAMPLES_PER_TYPE = 2

legend_patches = [
    mpatches.Patch(color=CLASS_COLORS[i] / 255.0, label=CLASS_NAMES[i])
    for i in range(NUM_CLASSES)
]

for dtype in DEFECT_TYPES:
    type_results = [r for r in results if r['defect_type'] == dtype]
    samples      = type_results[:SAMPLES_PER_TYPE]

    fig, axes = plt.subplots(len(samples), 3,
                             figsize=(13, 4 * len(samples)))
    if len(samples) == 1:
        axes = axes[np.newaxis, :]

    fig.suptitle(f'Defect type: {dtype.upper()}', fontsize=13, fontweight='bold')

    for row, rec in enumerate(samples):
        img_gray   = (rec['_img_tensor'][:, :, 0] * 255).astype(np.uint8)
        pred_mask  = rec['_pred_mask']
        color_mask = colorize_mask(pred_mask)

        # Overlay: colour mask blended onto grayscale
        img_rgb = cv2.cvtColor(img_gray, cv2.COLOR_GRAY2RGB)
        overlay = img_rgb.copy()
        defect_region = pred_mask > 0
        overlay[defect_region] = (
            0.45 * img_rgb[defect_region].astype(np.float32) +
            0.55 * color_mask[defect_region].astype(np.float32)
        ).astype(np.uint8)

        axes[row, 0].imshow(img_gray, cmap='gray')
        axes[row, 0].set_title(f'{rec["image"]} — original', fontsize=10)
        axes[row, 0].axis('off')

        axes[row, 1].imshow(color_mask)
        axes[row, 1].set_title(
            f'Predicted mask  |  defect: {rec["defect_pct"]:.2f}%  '
            f'({rec["defect_px"]:,} px)', fontsize=10)
        axes[row, 1].axis('off')

        axes[row, 2].imshow(overlay)
        axes[row, 2].set_title('Overlay', fontsize=10)
        axes[row, 2].axis('off')

    fig.legend(handles=legend_patches, loc='lower center',
               ncol=NUM_CLASSES, fontsize=9, framealpha=0.8)
    plt.tight_layout(rect=[0, 0.04, 1, 1])
    plt.show()

## 10. Export results to CSV

In [ ]:
OUT_DIR = r'd:\7th sem\Image processing and CV\Leather_defect_detection\unet_results'
os.makedirs(OUT_DIR, exist_ok=True)

csv_path = os.path.join(OUT_DIR, 'damage_area_report.csv')
df.to_csv(csv_path, index=False)
print(f'Full per-image report saved to: {csv_path}')

summary_path = os.path.join(OUT_DIR, 'damage_area_summary.csv')
summary.to_csv(summary_path, index=False)
print(f'Summary saved to:              {summary_path}')